In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from torch.utils.data import DataLoader
import os
import pandas as pd

import utils

data_path = utils.check_cifar_dataset_exists()
utils.check_mnist_dataset_exists()

device = torch.device('cuda')

### Child Network Class

In [ ]:
class ChildModel(nn.Module):
    def __init__(self, encoding, input_dim, output_dim):
        super(ChildModel, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.model = self.build_model_from_encoding(encoding)

    def forward(self, x):
        return self.model(x)


    def build_model_from_encoding(self, encoding):
        layers = []  # list ot store layers 
        sticky_dim = self.input_dim
        # TODO: If START found elsewhere give negative feedback and dont create model
        for layer in encoding: 
            layer_param = layer  # Get the type of the layer
            if layer_param.isnumeric():
                layer_param = 2 ** (int(layer_param) + 4)
                layers.append(nn.Linear(sticky_dim, layer_param))
                layers.append(nn.ReLU())
                sticky_dim = layer_param
            if layer_param == "END":
                layers.append(nn.Linear(sticky_dim, self.output_dim))
                # layers.append(nn.Softmax(dim=1))

        return nn.Sequential(*layers)  # Return the model


    def train_model(self, data, label, criterion, optimizer, device, epochs=10):
        self.model.train()  
        total_loss = 0  # Total loss of the model
        start_time = time.time()  # Start time of the training
        bs = 200

        self.model.to(device)

        for _ in range(epochs):
            shuffled_indices=torch.randperm(data.size(0))
            num_batches = 0
            running_loss = 0

            for iter in range(1, len(data),bs):
                num_batches += 1

                # Set dL/dU, dL/dV, dL/dW to be filled with zeros
                optimizer.zero_grad()

                # create a minibatch
                indices = shuffled_indices[iter:iter+bs]
                minibatch_data = data[indices]
                minibatch_label = label[indices]

                # send batch to device
                minibatch_data = minibatch_data.to(device)
                minibatch_label = minibatch_label.to(device)

                # reshape the minibatch
                inputs = minibatch_data.view(-1, self.input_dim)

                # tell Pytorch to start tracking all operations that will be done on "inputs"
                inputs.requires_grad_()

                # forward the minibatch through the net
                scores = self.model(inputs)

                # Compute the average of the losses of the data points in the minibatch
                loss = criterion(scores, minibatch_label)
                running_loss += loss.detach().item()

                # backward pass to compute dL/dU, dL/dV and dL/dW
                loss.backward()

                # do one step of stochastic gradient descent: U=U-lr(dL/dU), V=V-lr(dL/dU), ...
                optimizer.step()


            total_loss = running_loss / num_batches

        elapsed_time = time.time() - start_time
        return total_loss, elapsed_time


    def evaluate_model(self, data, labels, device):
        self.model.eval()

        bs = 200
        correct = 0
        total = 0

        with torch.no_grad():
            for i in range(0, data.size(0), bs):

                # Slice the batch manually
                minibatch_data = data[i:i+bs].to(device)
                minibatch_labels = labels[i:i+bs].to(device)

                inputs = minibatch_data.view(-1, self.input_dim)

                # Forward pass
                scores = self.model(inputs)
                predicted = torch.argmax(scores, dim=1)

                # Count correct predictions
                total += minibatch_labels.size(0)
                correct += torch.sum(predicted == minibatch_labels).item()

        return correct / total



### RNN Controller

In [ ]:
# Define vocabulary
MAX_NEURONS = 6
VOCAB = {"END": 0, **{str(i): i for i in range(1, MAX_NEURONS + 1)}}  # 1-10 neurons
IDX_TO_TOKEN = {v: k for k, v in VOCAB.items()}
VOCAB_SIZE = len(VOCAB)


class RNNController(nn.Module):
    def __init__(self, vocab_size, embedding_dim=5, hidden_dim=10, num_layers=1):
        super(RNNController, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)  # Output size = vocab_size

    def forward(self, x, hidden=None):
        x = self.embedding(x)  
        output, hidden = self.rnn(x, hidden) 
        logits = self.fc(output) 
        return logits, hidden


    # Function to sample a sequence
    def generate_sequence(self, max_layers=5):
        self.eval()
        start_token = torch.zeros(1, 1, dtype=torch.long)
        hidden = None
        sequence = ["START"]
        input_token = start_token
        log_probs = []

        for _ in range(max_layers):
            logits, hidden = self.forward(input_token, hidden)

            distribution = torch.distributions.Categorical(logits=logits)  # Convert logits to probability distribution
            token = distribution.sample()  # Sample a token
            log_prob = distribution.log_prob(token)  

            sequence.append(IDX_TO_TOKEN[token.item()])  
            log_probs.append(log_prob)

            if token == VOCAB["END"]:
                break
            input_token = torch.tensor([[token]])

        if sequence[-1] != "END":
            sequence.append("END")
        return sequence, torch.stack(log_probs)

### Model Generation and Training

In [ ]:
test_controller = RNNController(VOCAB_SIZE)

test_sequence, _ = test_controller.generate_sequence(max_layers=5)

print(test_sequence)

test_model = ChildModel(test_sequence, 3072, 10).to(device)
print(test_model)

test_model_loss, train_time = test_model.train_model(
    torch.load(data_path + 'cifar/train_data.pt'),
    torch.load(data_path + 'cifar/train_label.pt'),
    nn.CrossEntropyLoss(),
    torch.optim.SGD(test_model.parameters(), lr=0.001),
    device,
    epochs=10
)
print("child model final epoch loss:", test_model_loss, "with train time:", train_time)

test_accuracy = test_model.evaluate_model(
    torch.load(data_path + 'cifar/train_data.pt'),
    torch.load(data_path + 'cifar/train_label.pt'),
    device
)
print("child model validation set accuracy:", test_accuracy)

### RL Loop

In [13]:
def is_valid_encoding(encoding):
    if len(encoding) < 3 or encoding[0] != "START" or encoding[-1] != "END":
        return False

    for i in range(1, len(encoding) - 1):
        if not encoding[i].isnumeric():
            return False

    return True

def split_dataset(data, labels, split_ratio=0.8):
    dataset = torch.utils.data.TensorDataset(data, labels)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    train_set, test_set = torch.utils.data.random_split(dataset, [train_size, test_size])

    train_data, train_labels = zip(*train_set)
    train_data = torch.stack(train_data)
    train_labels = torch.stack(train_labels)

    test_data, test_labels = zip(*test_set)
    test_data = torch.stack(test_data)
    test_labels = torch.stack(test_labels)

    return (train_data, train_labels), (test_data, test_labels)

In [ ]:
def train_controller(controller, train, test=None, model_iters=5, train_epochs=10, negative_reward=-1, max_child_layers=5):


    if not test:
        train, test = split_dataset(*train)

    train_x, train_y = train

    test_x, test_y = test

    input_dim = train_x[0].view(1, -1).size()[1]
    output_dim = torch.max(train_y).item() + 1

    optimizer = torch.optim.Adam(controller.parameters(), lr=0.001)
    gamma = 0.9

    baseline = 0

    for iter in range(model_iters):
        print(f"------ ITERATION {iter} ------")

        model_encoding, log_probs = controller.generate_sequence(max_layers=max_child_layers)
        print("Model encoding:", model_encoding)

        child_model = ChildModel(model_encoding, input_dim, output_dim).to(device)

        print(child_model)

        # Train child model
        child_model_loss, train_time = child_model.train_model(
            train_x,
            train_y,
            nn.CrossEntropyLoss(),
            torch.optim.SGD(child_model.parameters(), lr=0.01),
            device,
            epochs=train_epochs
        )
        print("CHILD MODEL LOSS:", child_model_loss, "with train time:", train_time)

        accuracy = child_model.evaluate_model(test_x, test_y, device)
        print("CHILD MODEL ACCURACY:", accuracy)

        reward = accuracy

        # advantage = torch.tensor(reward - baseline, dtype=torch.float32)
        # advantage = torch.clamp(advantage, -0.1, 0.1)

        policy_gradient = -torch.sum(log_probs) * reward

        # baseline = (1 - gamma) * reward + gamma * baseline

        optimizer.zero_grad()
        policy_gradient.backward()
        optimizer.step()

        print("Policy gradient:", policy_gradient.item())

        print()

# Is It Okay If Higher Accuracy → Higher Loss (policy gradient)?
#
# Yes, because you're minimizing negative reward-weighted log-probabilities:
#
# Higher reward = higher positive loss value = stronger update in backward()
# Lower reward = smaller or even negative advantage = smaller gradient or a push in the opposite direction
# Loss value increasing is not a problem — because you’re not minimizing “loss” in the usual supervised learning sense — you're maximizing expected reward through gradient ascent via REINFORCE.


In [ ]:
train_data = (torch.load(data_path + 'cifar/train_data.pt'), torch.load(data_path + 'cifar/train_label.pt'))
test_data = (torch.load(data_path + 'cifar/test_data.pt'), torch.load(data_path + 'cifar/test_label.pt'))

train_controller(RNNController(VOCAB_SIZE), train_data, test_data, model_iters=20, train_epochs=50, negative_reward=-1, max_child_layers=5)

In [ ]:
train_data = (torch.load(data_path + 'mnist/train_data.pt'), torch.load(data_path + 'mnist/train_label.pt'))
test_data = (torch.load(data_path + 'mnist/train_data.pt'), torch.load(data_path + 'mnist/train_label.pt'))

train_controller(RNNController(VOCAB_SIZE), train_data, test_data, model_iters=20, train_epochs=10, negative_reward=-1, max_child_layers=3)

In [14]:
# Simple functions to load datasets as data, labels pairs. Return type should be tensors

def load_music_dataset():
    if not os.path.exists('data/music_genre_classification.csv'):
        print("Download from Kaggle: https://www.kaggle.com/datasets/purumalgi/music-genre-classification")
        raise FileNotFoundError('data/music_genre_classification.csv')

    df = pd.read_csv('data/music_genre_classification.csv')
    df = df.dropna()

    # Artist embeddings
    artist_to_id = {artist: idx for idx, artist in enumerate(df['Artist Name'].unique())}
    df['artist_id'] = df['Artist Name'].map(artist_to_id)
    artist_embedding_layer = nn.Embedding(len(artist_to_id), 10)
    artist_embeddings = artist_embedding_layer(torch.LongTensor(df['artist_id'].values)).detach()

    # Track embeddings
    track_to_id = {track: idx for idx, track in enumerate(df['Track Name'].unique())}
    df['track_id'] = df['Track Name'].map(track_to_id)
    track_embedding_layer = nn.Embedding(len(track_to_id), 10)
    track_embeddings = track_embedding_layer(torch.LongTensor(df['track_id'].values)).detach()

    # Normalize numerical features using PyTorch
    numerical_cols = [
        'Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode',
        'speechiness', 'acousticness', 'instrumentalness', 'liveness',
        'valence', 'tempo', 'duration_in min/ms', 'time_signature'
    ]
    x = torch.tensor(df[numerical_cols].values, dtype=torch.float32)
    mean = x.mean(dim=0, keepdim=True)
    std = x.std(dim=0, keepdim=True)
    std[std == 0] = 1.0  # Avoid divide-by-zero
    normalized_x = (x - mean) / std

    # Final dataset
    data = torch.hstack((artist_embeddings, track_embeddings, normalized_x))
    labels = torch.LongTensor(df['Class'].values)

    return data, labels



load_music_dataset()

Download from Kaggle: https://www.kaggle.com/datasets/purumalgi/music-genre-classification


FileNotFoundError: data/music_genre_classification.csv

In [ ]:
data, labels = load_music_dataset()
tx, ty = split_dataset(data, labels)
print(data.shape, labels.shape)
print(tx[0].shape, tx[1].shape, ty[0].shape, ty[1].shape)

In [ ]:
train_controller(RNNController(VOCAB_SIZE), (data, labels), model_iters=20, train_epochs=10, negative_reward=-1, max_child_layers=2)